# Run the full workflow

In [ ]:
import datetime

from st.dto.data import PriceDataDTO, ReturnsDTO, CorrelationDTO
from st.dto.volatility import StandardVolatilityDTO, EWMAVolatilityDTO, RobustVolatilityDTO, \
    VolatilityStandardizationDTO, VolatilityDTO, SimpleVolatilityForecastDTO, EWMAVolatilityForecastDTO
from st.plotter import PriceDataPlotter, ReturnsPlotter, VolatilityPlotter  # noqa

## Price Data, Returns, Correlation Matrix---

In [ ]:
# load price data
tickers = ["AAPL", "GOOGL", "AMZN", "MSFT"]
price_datas = [PriceDataDTO(ticker=tkr, start_date=datetime.datetime(year=2023, month=1, day=1)) for tkr in tickers]

# calculate returns
returns = [ReturnsDTO(price_data=pdata) for pdata in price_datas]

# calculate correlations
correlations = CorrelationDTO(price_datas=price_datas)

In [ ]:
for pdata in price_datas: print(pdata)
for rt in returns: print(rt)
print(correlations)
correlations.correlation_matrix

## Plotter for PriceData
---

In [ ]:
plotter = PriceDataPlotter()
for pdata in price_datas:
    plotter.add(pdata)

print(plotter)
plotter.show()

## Plotter for returns
---

In [ ]:
plotter = ReturnsPlotter()
for rt in returns:
    print(rt)
    plotter.add(rt)
print(plotter)
plotter.show(plot_type="both", width=900, height=600)

In [ ]:
plotter.show_distribution()

In [ ]:
plotter.show_skew_comparison()

## Volatility

- all the volatilises are similar.
- EWMA has sudden peaks (undesirable)
- Robust gives the lowest values most of the time. (undesirable)
- Standard seems a good one to use for volatility standardization.
---

In [ ]:
vols_classes = [StandardVolatilityDTO, EWMAVolatilityDTO, RobustVolatilityDTO]
vols_data: dict[str, list[VolatilityDTO]] = {}
for rt in returns:
    tkr = rt.price_data.ticker
    vols_data[tkr] = [vc(returns=rt) for vc in vols_classes]
    plotter = VolatilityPlotter()
    [plotter.add(i, tkr + i.__class__.__name__) for i in vols_data[tkr]]
    plotter.show()
vols_data

## Volatility Standardization
---

In [ ]:
# load price data
tickers = ["AAPL", "GOOGL", "AMZN", "MSFT"]
price_datas = [PriceDataDTO(ticker=tkr, start_date=datetime.datetime(year=2023, month=1, day=1)) for tkr in tickers]

print("Non Standardized Returns")
for rt in returns:
    print(rt)

# normal returns
rts = [ReturnsDTO(price_data=pdata) for pdata in price_datas]
rt_plotter = ReturnsPlotter()
for rt in rts:
    rt_plotter.add(rt)
rt_plotter.show()

print("Standardized Returns")

# volatility standardization
vol_st = [VolatilityStandardizationDTO(price_data=pdata) for pdata in price_datas]
standard_rts = [i.returns for i in vol_st]

plotter = VolatilityPlotter()
for v in vol_st:
    plotter.add(v.volatility)
plotter.show()

# plotting
plotter = ReturnsPlotter()
for rt in standard_rts:
    plotter.add(rt)

# for pdata in price_datas:
#     plotter.add(pdata)

plotter.show()

## Volatility Forecasts
---

In [ ]:
for tkr, v in vols_data.items():
    for vv in v:
        print(tkr, SimpleVolatilityForecastDTO(volatility=vv, method="last"))
    print()